# Day 09. Exercise 03
# Ensembles

## 0. Imports

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split

from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression

## 1. Preprocessing

1. Create the same dataframe as in the previous exercise.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test` and then get `X_train`, `y_train`, `X_valid`, `y_valid` from the previous `X_train`, `y_train`. Use the additional parameter `stratify`.

In [2]:
df = pd.read_csv('../data/day-of-week-not-scaled.csv', sep=',')
dow = pd.read_csv('../data/dayofweek.csv', sep=',')

X = df
y = dow['dayofweek']

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)

In [4]:
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.2, random_state=21, stratify=y_train)

## 2. Individual classifiers

1. Train SVM, decision tree and random forest again with the best parameters that you got from the 01 exercise with `random_state=21` for all of them.
2. Evaluate `accuracy`, `precision`, and `recall` for them on the validation set.
3. The result of each cell of the section should look like this:

```
accuracy is 0.87778
precision is 0.88162
recall is 0.87778
```

In [5]:
best_params_svm = {'kernel':'rbf', 'C':10, 'gamma':'auto', 'class_weight':None, 'probability':True}
best_params_tree = {'max_depth': 25,  'class_weight':'balanced', 'criterion':'entropy'}
best_params_forest = {'n_estimators': 100, 'max_depth': 26, 'class_weight':'balanced', 'criterion':'entropy'}

In [6]:
svm = SVC(**best_params_svm, random_state=21)
tree = DecisionTreeClassifier(**best_params_tree, random_state=21)
forest = RandomForestClassifier(**best_params_forest, random_state=21)

In [7]:
svm.fit(X_train, y_train)
tree.fit(X_train, y_train)
forest.fit(X_train, y_train)

,n_estimators,100
,criterion,'entropy'
,max_depth,26
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [8]:
models = {
    'SVM': svm,
    'Decision Tree': tree,
    'Random Forest': forest
}

for name, model in models.items():
    y_pred = model.predict(X_valid)
    acc = accuracy_score(y_valid, y_pred)
    prec = precision_score(y_valid, y_pred, average='weighted')
    rec = recall_score(y_valid, y_pred, average='weighted')
    print(f"{name}:")
    print(f"accuracy is {acc:.5f}")
    print(f"precision is {prec:.5f}")
    print(f"recall is {rec:.5f}")

SVM:
accuracy is 0.87778
precision is 0.88162
recall is 0.87778
Decision Tree:
accuracy is 0.85185
precision is 0.85481
recall is 0.85185
Random Forest:
accuracy is 0.89259
precision is 0.89230
recall is 0.89259


## 3. Voting classifiers

1. Using `VotingClassifier` and the three models that you have just trained, calculate the `accuracy`, `precision`, and `recall` on the validation set.
2. Play with the other parameteres.
3. Calculate the `accuracy`, `precision` and `recall` on the test set for the model with the best weights in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).

- Вместо того, чтобы выбирать одну из моделей, можно объединить их ответы и сделать финальное решение на основе «голосования».

Есть два основных режима:
- Hard voting (жёсткое голосование): каждая модель делает предсказание (метка класса). Финальный ответ — тот класс, который получил большинство голосов. 
- Soft voting (мягкое голосование): каждая модель выдаёт вероятности для каждого класса (если поддерживает predict_proba), ты усредняешь/взвешиваешь эти вероятности, и выбираешь класс с наибольшей суммарной вероятностью. 


Также есть параметр weights, с которым можно дать разным моделям разный «вес» в голосовании. То есть если ты считаешь, что одна модель более надёжна — можешь ей дать больший вес.

In [9]:
svm = SVC(
    kernel='rbf',
    C=10,
    gamma='auto',
    class_weight=None,
    probability=True,
    random_state=21
)

tree = DecisionTreeClassifier(
    max_depth=25,
    class_weight='balanced',
    criterion='entropy',
    random_state=21
)

forest = RandomForestClassifier(
    n_estimators=100,
    max_depth=25,
    class_weight='balanced',
    criterion='entropy',
    random_state=21
)

In [10]:
param_grid = [
    {'voting': 'soft', 'weights': [1, 1, 1]},
    {'voting': 'soft', 'weights': [1, 1, 2]},
    {'voting': 'soft', 'weights': [1, 2, 1]},
    {'voting': 'soft', 'weights': [2, 1, 1]},
    {'voting': 'hard', 'weights': [1, 1, 1]},
    {'voting': 'soft', 'weights': [4, 1, 4]}
]

In [11]:
results = []

for params in param_grid:
    voting = VotingClassifier(
        estimators=[
            ('svm', svm),
            ('tree', tree),
            ('forest', forest)
        ],
        voting=params['voting'],
        weights=params['weights']
    )

    voting.fit(X_train, y_train)
    y_pred = voting.predict(X_valid)

    acc = accuracy_score(y_valid, y_pred)
    prec = precision_score(y_valid, y_pred, average='weighted')
    rec = recall_score(y_valid, y_pred, average='weighted')

    results.append({
        'voting': params['voting'],
        'weights': params['weights'],
        'accuracy': acc,
        'precision': prec,
        'recall': rec
    })

In [12]:
results

[{'voting': 'soft',
  'weights': [1, 1, 1],
  'accuracy': 0.8851851851851852,
  'precision': 0.8872461686681917,
  'recall': 0.8851851851851852},
 {'voting': 'soft',
  'weights': [1, 1, 2],
  'accuracy': 0.8962962962962963,
  'precision': 0.8977606273507913,
  'recall': 0.8962962962962963},
 {'voting': 'soft',
  'weights': [1, 2, 1],
  'accuracy': 0.8518518518518519,
  'precision': 0.8548103632478633,
  'recall': 0.8518518518518519},
 {'voting': 'soft',
  'weights': [2, 1, 1],
  'accuracy': 0.9,
  'precision': 0.9015552240914727,
  'recall': 0.9},
 {'voting': 'hard',
  'weights': [1, 1, 1],
  'accuracy': 0.9074074074074074,
  'precision': 0.9071885521885522,
  'recall': 0.9074074074074074},
 {'voting': 'soft',
  'weights': [4, 1, 4],
  'accuracy': 0.9148148148148149,
  'precision': 0.9164162275079524,
  'recall': 0.9148148148148149}]

In [13]:
for r in results:
    print(f"Voting: {r['voting']}, weights: {r['weights']}")
    print(f"accuracy is {r['accuracy']:.5f}")
    print(f"precision is {r['precision']:.5f}")
    print(f"recall is {r['recall']:.5f}\n")

best_model = max(results, key=lambda x: (x['accuracy'], x['precision']))

print("Best model on validation set:")
print(best_model)

Voting: soft, weights: [1, 1, 1]
accuracy is 0.88519
precision is 0.88725
recall is 0.88519

Voting: soft, weights: [1, 1, 2]
accuracy is 0.89630
precision is 0.89776
recall is 0.89630

Voting: soft, weights: [1, 2, 1]
accuracy is 0.85185
precision is 0.85481
recall is 0.85185

Voting: soft, weights: [2, 1, 1]
accuracy is 0.90000
precision is 0.90156
recall is 0.90000

Voting: hard, weights: [1, 1, 1]
accuracy is 0.90741
precision is 0.90719
recall is 0.90741

Voting: soft, weights: [4, 1, 4]
accuracy is 0.91481
precision is 0.91642
recall is 0.91481

Best model on validation set:
{'voting': 'soft', 'weights': [4, 1, 4], 'accuracy': 0.9148148148148149, 'precision': 0.9164162275079524, 'recall': 0.9148148148148149}


In [14]:
final_voting = VotingClassifier(
    estimators=[
        ('svm', svm),
        ('tree', tree),
        ('forest', forest)
    ],
    voting=best_model['voting'],
    weights=best_model['weights']
)

final_voting.fit(X_train, y_train)
y_pred_test = final_voting.predict(X_test)

acc_test = accuracy_score(y_test, y_pred_test)
prec_test = precision_score(y_test, y_pred_test, average='weighted')
rec_test = recall_score(y_test, y_pred_test, average='weighted')

print(f"accuracy is {acc_test:.5f}")
print(f"precision is {prec_test:.5f}")
print(f"recall is {rec_test:.5f}")

accuracy is 0.90828
precision is 0.91059
recall is 0.90828


## 4. Bagging classifiers

1. Using `BaggingClassifier` and `SVM` with the best parameters create an ensemble, try different values of the `n_estimators`, use `random_state=21`.
2. Play with the other parameters.
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision)

Bagging (Bootstrap Aggregating) — это способ уменьшить разброс (variance) модели и повысить устойчивость предсказаний.
- создаются несколько подвыборок из исходных данных (с возвращением)
- на каждой подвыборке обучается своя копия модели
- финальный результат — усреднение (для регрессии) или голосование (для классификации) между этими моделями.

In [15]:
base_svm = SVC(
    kernel='rbf',
    C=10,
    gamma='auto',
    class_weight=None,
    probability=True,
    random_state=21
)

In [16]:
param_grid = [
    {'n_estimators': 5, 'max_samples': 0.8, 'max_features': 1.0, 'bootstrap': True},
    {'n_estimators': 10, 'max_samples': 0.8, 'max_features': 1.0, 'bootstrap': True},
    {'n_estimators': 20, 'max_samples': 0.9, 'max_features': 0.8, 'bootstrap': True},
    {'n_estimators': 30, 'max_samples': 1.0, 'max_features': 1.0, 'bootstrap': False},
    {'n_estimators': 50, 'max_samples': 0.9, 'max_features': 1.0, 'bootstrap': True},
]

- n_estimators 5 — количество моделей внутри ансамбля (т.е. будет 5 SVM, каждая обучена на разных частях данных).
- max_samples 0.8 — каждая модель видит только 80% обучающей выборки (случайно выбранной с возвращением, если bootstrap=True) (разнообразие, снижает переобучение)
- max_features 1.0 — каждая модель использует все признаки (100%). (Можно уменьшить, чтобы каждая модель видела разные части признаков)
- bootstrap=True — выборка строится с возвращением (то есть некоторые объекты могут попасть несколько раз, некоторые — ни разу).(если поставить False, подвыборки будут без возвращения — меньше случайности) 

In [18]:
results = []

for params in param_grid:
    bagging = BaggingClassifier(
        estimator=base_svm,
        n_estimators=params['n_estimators'],
        max_samples=params['max_samples'],
        max_features=params['max_features'],
        bootstrap=params['bootstrap'],
        random_state=21
    )

    bagging.fit(X_train, y_train)
    y_pred = bagging.predict(X_valid)

    acc = accuracy_score(y_valid, y_pred)
    prec = precision_score(y_valid, y_pred, average='weighted')
    rec = recall_score(y_valid, y_pred, average='weighted')

    results.append({
        'n_estimators': params['n_estimators'],
        'max_samples': params['max_samples'],
        'max_features': params['max_features'],
        'bootstrap': params['bootstrap'],
        'accuracy': acc,
        'precision': prec,
        'recall': rec
    })

In [19]:
for r in results:
    print(f"n_estimators={r['n_estimators']}, max_samples={r['max_samples']}, "
          f"max_features={r['max_features']}, bootstrap={r['bootstrap']}")
    print(f"accuracy is {r['accuracy']:.5f}")
    print(f"precision is {r['precision']:.5f}")
    print(f"recall is {r['recall']:.5f}\n")

# Выбираем лучшую модель на валидации
best_model = max(results, key=lambda x: (x['accuracy'], x['precision']))
print("Best Bagging classifiers (validation):")
print(best_model)

n_estimators=5, max_samples=0.8, max_features=1.0, bootstrap=True
accuracy is 0.85926
precision is 0.87327
recall is 0.85926

n_estimators=10, max_samples=0.8, max_features=1.0, bootstrap=True
accuracy is 0.86667
precision is 0.87600
recall is 0.86667

n_estimators=20, max_samples=0.9, max_features=0.8, bootstrap=True
accuracy is 0.84444
precision is 0.86038
recall is 0.84444

n_estimators=30, max_samples=1.0, max_features=1.0, bootstrap=False
accuracy is 0.88889
precision is 0.89378
recall is 0.88889

n_estimators=50, max_samples=0.9, max_features=1.0, bootstrap=True
accuracy is 0.87407
precision is 0.88390
recall is 0.87407

Best Bagging classifiers (validation):
{'n_estimators': 30, 'max_samples': 1.0, 'max_features': 1.0, 'bootstrap': False, 'accuracy': 0.8888888888888888, 'precision': 0.8937816791667883, 'recall': 0.8888888888888888}


In [20]:
# На тестовой выборке 

final_bagging = BaggingClassifier(
    estimator=base_svm,
    n_estimators=best_model['n_estimators'],
    max_samples=best_model['max_samples'],
    max_features=best_model['max_features'],
    bootstrap=best_model['bootstrap'],
    random_state=21
)

final_bagging.fit(X_train, y_train)
y_pred_test = final_bagging.predict(X_test)

acc_test = accuracy_score(y_test, y_pred_test)
prec_test = precision_score(y_test, y_pred_test, average='weighted')
rec_test = recall_score(y_test, y_pred_test, average='weighted')

print("\nFinal Bagging model on test set:")
print(f"accuracy is {acc_test:.5f}")
print(f"precision is {prec_test:.5f}")
print(f"recall is {rec_test:.5f}")


Final Bagging model on test set:
accuracy is 0.88757
precision is 0.88906
recall is 0.88757


## 5. Stacking classifiers

1. To achieve reproducibility in this case you will have to create an object of cross-validation generator: `StratifiedKFold(n_splits=n, shuffle=True, random_state=21)`, where `n` you will try to optimize (the details are below).
2. Using `StackingClassifier` and the three models that you have recently trained, calculate the `accuracy`, `precision` and `recall` on the validation set, try different values of `n_splits` `[2, 3, 4, 5, 6, 7]` in the cross-validation generator and parameter `passthrough` in the classifier itself,
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision). Use `final_estimator=LogisticRegression(solver='liblinear')`.

bagging — “много одинаковых моделей”, stacking — “несколько разных моделей, обученных на одних данных”.

Как работает:
1. несколько разных моделей (нSVM, RandomForest, DecisionTreeClassifier).
2. Каждая из них делает свои предсказания.
3. Эти предсказания становятся новыми признаками.
4. На них обучается финальная модель — final_estimator (LogisticRegression)
- Её задача — “понять”, какой из базовых алгоритмов лучше предсказывает в какой ситуации.
- Если passthrough=True, то финальная модель видит и исходные признаки, и предсказания базовых моделей — иногда это повышает качество, особенно если базовые модели не идеально согласованы.

In [21]:
svm = SVC(kernel='rbf', C=10, gamma='auto', class_weight=None, probability=True, random_state=21)
tree = DecisionTreeClassifier(max_depth=25, class_weight='balanced', criterion='entropy', random_state=21)
forest = RandomForestClassifier(n_estimators=100, max_depth=26, class_weight='balanced', criterion='entropy', random_state=21)

In [22]:
final_estimator = LogisticRegression(solver='liblinear', random_state=21)

# Параметры для перебора
n_splits_list = [2, 3, 4, 5, 6, 7]
passthrough_options = [True, False]

results = []

# Перебор комбинаций
for n in n_splits_list:
    for passthrough in passthrough_options:
        cv_gen = StratifiedKFold(n_splits=n, shuffle=True, random_state=21)

        stacking = StackingClassifier(
            estimators=[('svm', svm), ('tree', tree), ('forest', forest)],
            final_estimator=final_estimator,
            cv=cv_gen,
            passthrough=passthrough
        )

        stacking.fit(X_train, y_train)
        y_pred = stacking.predict(X_valid)

        acc = accuracy_score(y_valid, y_pred)
        prec = precision_score(y_valid, y_pred, average='weighted')
        rec = recall_score(y_valid, y_pred, average='weighted')

        results.append({
            'n_splits': n,
            'passthrough': passthrough,
            'accuracy': acc,
            'precision': prec,
            'recall': rec
        })


c:\Users\Анастасия\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
c:\Users\Анастасия\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
c:\Users\Анастасия\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classifica

In [23]:
for r in results:
    print(f"n_splits={r['n_splits']}, passthrough={r['passthrough']}")
    print(f"accuracy is {r['accuracy']:.5f}")
    print(f"precision is {r['precision']:.5f}")
    print(f"recall is {r['recall']:.5f}\n")

# --- Выбор лучшей модели ---
best_model = max(results, key=lambda x: (x['accuracy'], x['precision']))
print("Best Stacking parameters (validation):")
print(best_model)

n_splits=2, passthrough=True
accuracy is 0.89630
precision is 0.89776
recall is 0.89630

n_splits=2, passthrough=False
accuracy is 0.90000
precision is 0.90077
recall is 0.90000

n_splits=3, passthrough=True
accuracy is 0.90370
precision is 0.90658
recall is 0.90370

n_splits=3, passthrough=False
accuracy is 0.90370
precision is 0.90610
recall is 0.90370

n_splits=4, passthrough=True
accuracy is 0.91111
precision is 0.91409
recall is 0.91111

n_splits=4, passthrough=False
accuracy is 0.89630
precision is 0.89955
recall is 0.89630

n_splits=5, passthrough=True
accuracy is 0.90741
precision is 0.90954
recall is 0.90741

n_splits=5, passthrough=False
accuracy is 0.89259
precision is 0.89440
recall is 0.89259

n_splits=6, passthrough=True
accuracy is 0.90741
precision is 0.90944
recall is 0.90741

n_splits=6, passthrough=False
accuracy is 0.89630
precision is 0.89814
recall is 0.89630

n_splits=7, passthrough=True
accuracy is 0.91111
precision is 0.91264
recall is 0.91111

n_splits=7, pass

In [24]:
best_cv = StratifiedKFold(n_splits=best_model['n_splits'], shuffle=True, random_state=21)
final_stacking = StackingClassifier(
    estimators=[('svm', svm), ('tree', tree), ('forest', forest)],
    final_estimator=final_estimator,
    cv=best_cv,
    passthrough=best_model['passthrough']
)

final_stacking.fit(X_train, y_train)
y_pred_test = final_stacking.predict(X_test)

acc_test = accuracy_score(y_test, y_pred_test)
prec_test = precision_score(y_test, y_pred_test, average='weighted')
rec_test = recall_score(y_test, y_pred_test, average='weighted')

print("\nFinal Stacking model on test set:")
print(f"accuracy is {acc_test:.5f}")
print(f"precision is {prec_test:.5f}")
print(f"recall is {rec_test:.5f}")


Final Stacking model on test set:
accuracy is 0.90828
precision is 0.90967
recall is 0.90828


c:\Users\Анастасия\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(


## 6. Predictions

1. Choose the best model in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).
2. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which labname and for which users.
3. Save the model.

In [32]:
best_final_voting = VotingClassifier(
    estimators=[
        ('svm', svm),
        ('tree', tree),
        ('forest', forest)
    ],
    voting='soft',
    weights=[4,1,4]
)

best_final_voting.fit(X_train, y_train)
y_pred_test = best_final_voting.predict(X_test)

In [33]:
test_data = X_test.copy()
test_data['true'] = y_test
test_data['pred'] = y_pred_test

test_data['error'] = test_data['true'] != test_data['pred']

In [34]:
lab_cols = [c for c in X.columns if c.startswith('labname_')]
user_cols = [c for c in X.columns if c.startswith('uid_user_')]

test_data['labname'] = test_data[lab_cols].idxmax(axis=1).str.replace('labname_', '')
test_data['user'] = test_data[user_cols].idxmax(axis=1).str.replace('uid_user_', '')

In [35]:
weekday_errors = (test_data.groupby('true')['error'].mean() * 100).sort_values(ascending=False)
weekday_errors

true
0    29.629630
5    14.814815
1     9.090909
6     8.450704
2     6.666667
4     4.761905
3     3.750000
Name: error, dtype: float64

In [36]:
lab_errors = (
    test_data.groupby('labname')['error']
    .mean()
    .sort_values(ascending=False) * 100
)

lab_errors

labname
lab03       100.000000
laba04       28.571429
laba04s      24.000000
lab05s       16.666667
laba06       11.111111
code_rvw      7.692308
laba06s       6.666667
project1      6.451613
lab03s        0.000000
laba05        0.000000
Name: error, dtype: float64

In [30]:
user_errors = (
    test_data.groupby('user')['error']
    .mean()
    .sort_values(ascending=False) * 100
)

user_errors

user
16    40.000000
6     25.000000
3     21.428571
18    16.666667
14    16.129032
17    14.285714
30    12.500000
13    11.764706
31    11.111111
2     10.714286
19    10.526316
25     9.090909
24     9.090909
29     9.090909
10     8.333333
4      7.407407
21     7.142857
1      0.000000
12     0.000000
15     0.000000
27     0.000000
22     0.000000
23     0.000000
20     0.000000
26     0.000000
28     0.000000
8      0.000000
Name: error, dtype: float64

In [37]:
joblib.dump(best_final_voting, '../data/best_final_voting.pkl')


['../data/best_final_voting.pkl']